<!-- dd:dd-lesson-np-2 -->

# Indexing and selection

*Numpy · `np-2`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-2"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            if case.get("assert_code"):
                exec(case["assert_code"], dict(ns, result=actual))
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrVWu1uozoQfRUr0tXCLnD9gTGs1L/3CfZfNqrYhO5GSpMKaNVt1Xe/Htvho01acAg0UZsQIHjO8Yx9ZuznGWXR7Dt6nt1s"
    "5ces2G0espmHZsu0yAp5Zv48K7Ly/u56uVtlcMf69m6Xl6jc5cs/KC1Q+XP7iK5QGZTZttjlznxOAuwhDm80wAsPzQUc44B7"
    "iMkTC/fndnlf7m5u5M9owHVrmw083Nmkt79WKcq/I2ddrLdFmW6XmZN78vk/1PNdDzlFmTtlkBbXpsncDVbl37vMhdtap8vd"
    "Zl2UjitfjoLmPHpIN+660HD2eJcty2x1LQ9yZcGP/D6TbXxRAINfu93mi4Qw/y/dFPK8vqq+ALLGV31NonNnLx7qzxoDkljA"
    "4bH0IFfyxMVxBWjMV02aNT8+BYJ8rBnyGVCEtX/5yuP8MKC8zRhcuGTKhmUwBJYUVRG8ha/9K7wwsg6EpDU3+DUZ0nUuZGSq"
    "h52FhD7j9NTh/KlFzZshzril242d9bZ01bsEXUF9eh8i69qJbUtbUXJG87CdeYo19BWJM5omTmDu3JQp7yQDeWeap9vfmUOI"
    "2z9Ij0Sih54OROUHwJw5jKlS6UivhSlI/st5yY/kv/BQ7KFEXlaTVPM+eZu8S97UvMe16zumHiGfniz6c3F0UOqGvt24jfU1"
    "ExNY3+4uSwS+hkCo6k06IQwwwTc22CCJprAdGtVDQ8yHnbgg4uAPgo+oQ2ricH/WROJiAtxNixpm2HTbfA9HfU6AxdZuiDyi"
    "0h0gYopOeGOBLf9iIuqbbesooowOm84zNbwlenghIYjkFSSjo4vjVT9dfES3VvrQLlFQUwVXsSv0rK3p4J+djmNE7NW8HR1i"
    "j59eBH5bmKGSKFLpELIHTC7D/1+/92Vgr7MldGaQhxfi6kerBD0v1ak1CU+WKGnLr4iu+elKqamZQllmgIlE8i1TtCg8MKmk"
    "H0wqsrNZ57mwDWiPoEY1MRZLHLjqEzx9d/TC4DhV0FJpvgqzqzK42ezSklEX+XLSctG/SFqSZ8Wf9A7uu8g+IlUfUVPY/AS+"
    "pjUYHrgGByEJfyaVgSICHIYqcQjrrGayVGZvUcMMKyktoBAAK0bw+dlTmTrSmAypfxBH3xCuw4pgVcmcIiUgrcwSv3UbbJzJ"
    "lKMa9aleP7Pt5v2PR+emslrFKYtPDdOHtkysBiOhjyBxemdYMjVUY/zDEeNJR47btqj5169lhf/hbHZOa8wKnVq089VKsC9g"
    "te40W5iVLSI4tV3cq918d79dOY15OT4wLX9DBGZlEtXjR+yirwgGkVW2XN9KPXrFTjZbT08kGbhEoDqXaMfnw8i+Wv9/VIKu"
    "G8e15rTJ8hItI/g+WFQ4x0EyHhZ82II4gVdEYs4wFcwSnh8apriKxBG7SLeMzRBg2z31WMaH0nudEdAKQKNtq4LJe6PeOfyp"
    "muySeFhNigMoYQegCoJYHXGojZRp/jsr5Z246/r54fVEiVk96SwrxaTyRdgL0bbb57jrPohTLCeWlpvcumUyHcNgy1Vv8Ype"
    "MYZb1MvNw+dhiZHDQinjSTIuCDxb6a1zRmIUPwMwEyBQiYQlArViC0udCXxOwr913gO0xyqvmoh13Nd1GuUkDkkuwe0sV3YG"
    "Hx+JqXyY+kdVdwmHDXc16ydDFWU7g6vFRhjYxjmv9vrxasYYy358qG27rRXnUqxPZ1KsrzZUDCr2nrqJvR522memXZLS8XZg"
    "BTgiKkWKkzgSlGAlSglVO4RJLNQ1xiMaJTGL4KS+xAjF8IpCJhgmkfoZE+qaPPX2Z3qbxWjN8XrpKSLD7pvD2NX1iYgFYjAP"
    "9dDD+50VhXaRRHUiXIl2rlWlAmCxa9naftnu6br9oY9kH8Jq2bjlACxqnqXLjGkzNF2p+CgeXseDcKl1/CSrDdR+r948MeX6"
    "KWTkCWV/Nl3OwWwXdyjo3qQte+XYHY8PQeglGOM2i5eX/wEqKXAF"
)
print("Delta Drills checker ready — 14 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-boolean-masking -->

## Boolean masks — compare, count, filter

`numpy.boolean-masking`


<!-- dd:dd-seg-numpy-boolean-masking-0 -->

### the comparison IS the mask


A comparison applied to an array is itself elementwise: `x > 2.5` produces a
**boolean array** the same shape as `x` — `True` where the condition holds.
That boolean array is called a **mask**.

Saying "the comparison IS the mask" kills the urge to loop: there is no
separate "test each element" step to write. Divisibility, sign, range
membership, "equal to any of…" — anything you can phrase as an elementwise
condition becomes a mask in one expression.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nx = t.tensor([1, 4, 6, 9, 3, 7])\n\n# The comparison itself is the mask — same shape as x, dtype bool.\nmask = x > 5\n\n# Any elementwise condition works the same way, e.g. divisibility:\neven = x % 2 == 0\nprint("x      ", x)\nprint("x > 5  ", mask, mask.dtype)\nprint("x even ", even)\n# Hidden checks\nassert mask.tolist() == [False, False, True, True, False, True]\nassert even.tolist() == [False, True, True, False, False, False]\n', globals()), end='')




Why: one expression, no loop — the condition is written on the whole array
at once, and the result carries a True/False verdict per element.


<!-- dd:dd-q236 -->

### Problem 236 · faded — your turn

Boolean array marking entries strictly greater than a threshold.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[False,  True, False],
        [ True, False,  True]])
```


In [ ]:
import torch as t

def solve(x, threshold):
    """True exactly where x exceeds threshold."""
    return x _____ threshold


# Example run — the grader calls solve() with several different arrays
# and thresholds, including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 5.0, 2.0], [7.0, 0.5, 3.0]])
print(solve(example, 2.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(236)


In [ ]:
#@title 💡 Solution — Problem 236
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, threshold):
    return x > threshold


example = t.tensor([[1.0, 5.0, 2.0], [7.0, 0.5, 3.0]])
print(solve(example, 2.5))


<!-- dd:dd-seg-numpy-boolean-masking-1 -->

### using a mask — count and filter


Once you have a mask, two of its three uses are read-only:

- **count / reduce**: `t.count_nonzero(mask)` or `mask.sum()` (how many? —
  True behaves as 1), `mask.any()` / `mask.all()` (yes/no questions). Wrap in
  `int(...)` when a plain Python int is required.
- **filter**: `x[mask]` returns a 1-D array of just the selected elements
  (a *copy*, unlike slices) — however many there are, shape not preserved.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nx = t.tensor([1, 4, 6, 9, 3, 7])\nmask = x > 5\n\n# Count: True behaves as 1, so both spellings work.\n\n# Filter: mask indexing keeps just the True positions (as a copy).\nprint("count", int(mask.sum()), "| kept", x[mask])\n# Hidden checks\nassert t.count_nonzero(mask) == 3\nassert mask.sum() == 3\nassert x[mask].tolist() == [6, 9, 7]\n', globals()), end='')




Why: `count_nonzero`/`sum` on a mask is the standard "how many satisfy…?";
`x[mask]` is the standard "give me those entries".


<!-- dd:dd-q52 -->

### Problem 52 · faded — your turn

Number of True entries in a boolean array, as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3
```


In [ ]:
import torch as t

def solve(z):
    """Count the True entries of boolean array z."""
    return int(t._____(z))


# Example run — the grader calls solve() with several arrays.
example = t.tensor([True, False, True, True])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(52)


In [ ]:
#@title 💡 Solution — Problem 52
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return int(t.count_nonzero(z))


example = t.tensor([True, False, True, True])
print(solve(example))


<!-- dd:dd-seg-numpy-boolean-masking-2 -->

### combining masks and masked assignment


Masks combine with `&` (and), `|` (or), `~` (not) — NOT Python's
`and`/`or`/`not`, which fail on arrays. Because `&`/`|` bind tighter than
comparisons, each comparison needs parentheses: `(x > 3) & (x < 8)`.

The third use of a mask is **assignment**: `x[mask] = value` (or
`x[mask] *= -1`) rewrites only the selected positions, *in place*. That
mutates the original array, so the usual contract applies: "do not modify
the input" ⇒ `.copy()` first, then assign through the mask on the copy.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nx = t.tensor([1, 4, 6, 9, 3, 7])\n\n# Combined condition + masked assignment, on a copy to protect x.\n# Parentheses around EACH comparison are mandatory with & and |.\nout = x.clone()\nout[(out > 3) & (out < 8)] *= -1\nprint("(x > 3) & (x < 8) ->", (x > 3) & (x < 8))\nprint("out", out)\nprint("x  ", x, " <- untouched")\n# Hidden checks\nassert out.tolist() == [1, -4, -6, 9, 3, -7]\nassert x.tolist() == [1, 4, 6, 9, 3, 7]      # input untouched\n', globals()), end='')




Why: try removing the parentheses mentally: `out > 3 & out < 8` would
evaluate `3 & out` first (bitwise on ints!) — the precedence trap is why the
parenthesized form should become muscle memory.


<!-- dd:dd-q12 -->

### Problem 12 · faded — your turn

Entries strictly between 3 and 8 negated, input untouched.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0,  1,  2,  3, -4, -5, -6, -7,  8,  9, 10])
```


In [ ]:
import torch as t

def solve(z):
    """z with entries strictly between 3 and 8 negated (z unmodified)."""
    out = z.clone()
    out[(out > 3) _____ (out < 8)] *= -1
    return out


# Example run — the grader calls solve() with several arrays.
example = t.arange(11)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(12)


In [ ]:
#@title 💡 Solution — Problem 12
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    out = z.clone()
    out[(out > 3) & (out < 8)] *= -1
    return out


example = t.arange(11)
print(solve(example))


<!-- dd:dd-q85 -->

### Problem 85 · guided

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns a 2-D tensor containing only the rows of z that have at least one nonzero entry, in their original order. If every row is all zeros, return an empty tensor with zero rows.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[3, 0]])
```


<details>
<summary>Hints</summary>

1. Build the per-row test first: a row is dropped when EVERY entry is
   zero. That is a reduction over dim=1 producing one bool per row.
2. You want to keep the rows where that is false — negate the mask with
   `~`, then index the tensor with it.
3. `z[~(z == 0).all(dim=1)]` — an all-zero input drops to a (0, c) tensor
   by itself, which is exactly the required empty result.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the rows of z that are not entirely zero."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[0, 0], [3, 0], [0, 0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(85)


In [ ]:
#@title 💡 Solution — Problem 85
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z[~(z == 0).all(dim=1)]


example = t.tensor([[0, 0], [3, 0], [0, 0]])
print(solve(example))


<!-- dd:dd-q232 -->

### Problem 232 · independent

Write a function solve(x, d) that takes a 1-D PyTorch tensor of integers x and a positive integer d, and returns a boolean PyTorch tensor of the same shape. Entry i of the result must be True exactly when x[i] is divisible by d, and False otherwise. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ True, False,  True,  True, False])
```


In [ ]:
import torch as t

def solve(x, d):
    """Return a boolean array marking which elements of x are divisible by d."""
    return None


# Example run — the grader calls solve() with several different arrays and
# divisors, including edge cases. Your function must work for all of them.
example = t.tensor([3, 5, 9, 12, 14])
print(solve(example, 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(232)


In [ ]:
#@title 💡 Solution — Problem 232
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, d):
    return x % d == 0


example = t.tensor([3, 5, 9, 12, 14])
print(solve(example, 3))


<!-- dd:dd-q145 -->

### Problem 145 · independent

Write a function solve(a) that takes a 1-D float tensor and returns the indices of its LOCAL PEAKS: positions i (with 1 <= i <= len(a)-2) where a[i] is strictly greater than both immediate neighbors. The first and last positions can never qualify. Return an integer tensor (possibly empty), indices ascending.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 3])
```


In [ ]:
import torch as t

def solve(a):
    """Return indices of entries strictly greater than both neighbors."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([1.0, 3.0, 2.0, 5.0, 4.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(145)


In [ ]:
#@title 💡 Solution — Problem 145
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    return t.where((a[1:-1] > a[:-2]) & (a[1:-1] > a[2:]))[0] + 1


print(solve(t.tensor([1.0, 3.0, 2.0, 5.0, 4.0])))


<!-- dd:dd-q202 -->

### Problem 202 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns the subarray of rows that are NOT constant — rows containing at least two different values. Rows where every entry is equal are dropped; the rest keep their order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3]])
```


In [ ]:
import torch as t

def solve(z):
    """Return the rows of z that contain at least two distinct values."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 1, 1], [1, 2, 3]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(202)


In [ ]:
#@title 💡 Solution — Problem 202
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    mask = (z != z[:, :1]).any(dim=1)
    return z[mask]


print(solve(t.tensor([[1, 1, 1], [1, 2, 3]])))


#### Common mistakes

- **"Combine conditions with `and`/`or`."** — Those are Python's short-circuit
  operators and raise on arrays. Masks combine with `&`, `|`, `~` — and each
  comparison must be parenthesized because `&` binds tighter than `>`.
- **"`x[mask]` keeps the array's shape."** — It returns a 1-D array of just
  the selected elements, however many there are. Only masked *assignment*
  leaves the shape intact.
- **"Counting Trues needs a loop or list.count."** — A mask is 0s and 1s:
  `mask.sum()` or `t.count_nonzero(mask)`. For yes/no rather than how-many,
  `any()`/`all()`.


<!-- dd:dd-kp-numpy-argmin-argmax -->

## Locating extremes — argmin and argmax

`numpy.argmin-argmax`


<!-- dd:dd-seg-numpy-argmin-argmax-0 -->

### argmin/argmax — the index, not the value


`min`/`max` tell you the extreme **value**; the `arg` twins tell you **where
it lives**:

- **`t.argmin(v)` / `v.argmin()`** — index of the smallest element.
- **`t.argmax(v)` / `v.argmax()`** — index of the largest.

**Ties break to the first occurrence.** If the extreme value appears more
than once, you get the smallest index — deterministically. Many drills state
"replace only the first occurrence"; argmin/argmax gives exactly that for
free.

Like other reductions, the result is a 0-dim TENSOR, not a Python int;
wrap in `int(...)` when
a plain Python int is required. (Per-row/column argmax with `axis=` appears
in the broadcasting lesson — same idea, one axis at a time.)


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nv = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])\n\n# Index of the minimum. 2.0 appears twice — argmin reports the FIRST.\ni = int(t.argmin(v))\n\n# The value at that index is the min itself.\nprint("v", v)\nprint("argmin ->", i, "| v[i] =", v[i].item(), "| v.min() =", v.min().item())\n# Hidden checks\nassert i == 1\nassert v[i] == v.min()\n', globals()), end='')




Why: `int(...)` at the boundary again — graders asking for "a plain Python
int" reject `t.int64`. And `v[v.argmax()]` is how you get the value back
when you need both.


<!-- dd:dd-q38 -->

### Problem 38 · faded — your turn

Index of the smallest element (first occurrence on ties), as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
1
```


In [ ]:
import torch as t

def solve(v):
    """Index of v's smallest element, first occurrence on ties."""
    return int(t._____(v))


# Example run — the grader calls solve() with several different vectors,
# including edge cases. Your function must work for all of them.
example = t.tensor([4.0, 2.0, 7.0, 2.5, 9.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(38)


In [ ]:
#@title 💡 Solution — Problem 38
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    return int(t.argmin(v))


example = t.tensor([4.0, 2.0, 7.0, 2.5, 9.0])
print(solve(example))


<!-- dd:dd-seg-numpy-argmin-argmax-1 -->

### the index as a handle for surgery


"Replace the largest entry with 0" is: copy (if the input must survive), then
`out[out.argmax()] = 0`. One read, one write, no scanning loop — the index is
a *handle* you use to edit the array.

The habit to build: **protect the input, then operate.** Copy first, then
assign through the index. It reads cleanest and never backfires.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nv = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])\n\n# Replace the max with 0 on a copy: the index is the handle.\nout = v.clone()\nout[out.argmax()] = 0.0          # argmax -> 4; out[4] = 0\nprint("out", out)\nprint("v  ", v, " <- untouched")\n# Hidden checks\nassert out.tolist() == [4.0, 2.0, 7.0, 2.0, 0.0]\nassert v.tolist() == [4.0, 2.0, 7.0, 2.0, 9.0]   # input intact\n', globals()), end='')




Why: the copy-then-assign order matters when the input must survive —
`v[v.argmax()] = 0` would mutate the caller's array.


<!-- dd:dd-q219 -->

### Problem 219 · faded — your turn

Largest entry replaced with 0 (first occurrence only), input untouched.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 3., -1.,  0.,  2.])
```


In [ ]:
import torch as t

def solve(x):
    """x with its largest entry replaced by 0.0, without mutating x."""
    result = x._____()
    result[t._____(x)] = 0.0
    return result


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([3.0, -1.0, 7.5, 2.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(219)


In [ ]:
#@title 💡 Solution — Problem 219
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    result = x.clone()
    result[t.argmax(x)] = 0.0
    return result


example = t.tensor([3.0, -1.0, 7.5, 2.0])
print(solve(example))


<!-- dd:dd-seg-numpy-argmin-argmax-2 -->

### closest-to-target — argmin on a transformed array


argmin/argmax compose with transformed arrays. The pattern
`t.argmin(t.abs(z - target))` answers "which entry is *closest to*
target?" — build the quantity you want minimized, then ask where its minimum
sits. Any "closest / best / most-similar" task is this pattern with a
different transform.

Keep the roles straight: the *transformed* array chooses the index; the
*original* array supplies the value at that index.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nv = t.tensor([4.0, 2.0, 7.0, 2.0, 9.0])\n\n# "Closest to target" = argmin of a transformed array.\ntarget = 6.5\nj = int(t.argmin(t.abs(v - target)))\nclosest_value = v[j]\nprint("gaps to", target, ":", t.abs(v - target))\nprint("smallest gap at index", j, "-> value", closest_value.item())\n# Hidden checks\nassert j == 2                     # |7.0 - 6.5| = 0.5 is the smallest gap\nassert closest_value == 7.0\n', globals()), end='')




Why: no sorting needed — sorting is O(n log n) and loses positions;
`argmin(|v - t|)` is one pass and keeps them.


<!-- dd:dd-q98 -->

### Problem 98 · faded — your turn

The INDEX of the entry closest to target, as a plain int.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
1
```


In [ ]:
import torch as t

def solve(z, target):
    """Index of the entry of z closest to target."""
    return int(t._____(t._____(z - target)))


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([0.1, 0.4, 0.8]), 0.5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(98)


In [ ]:
#@title 💡 Solution — Problem 98
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, target):
    return int(t.abs(z - target).argmin())


print(solve(t.tensor([0.1, 0.4, 0.8]), 0.5))


<!-- dd:dd-q1 -->

### Problem 1 · guided

Write a function solve(z) that takes a 2-D PyTorch tensor z and returns a 1-D integer tensor containing, for each row, the index of that row's largest value. If a row's maximum appears more than once, report the first (leftmost) occurrence. The matrix can have any shape, including a single row or a single column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0])
```


<details>
<summary>Hints</summary>

1. Per row means the reduction runs ALONG the columns — which axis number
   is that for a 2-D tensor?
2. You want the position of the max, not the max itself, and torch already
   breaks ties leftmost.
3. `z.argmax(dim=1)` — first-occurrence tie-breaking is the documented
   default, so no extra work is needed.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the per-row index of the largest value in z."""
    return None


# Example run — the grader calls solve() with several different arrays.
example = t.tensor([[1, 9, 3], [7, 2, 5]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1)


In [ ]:
#@title 💡 Solution — Problem 1
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.argmax(dim=1)


example = t.tensor([[1, 9, 3], [7, 2, 5]])
print(solve(example))


<!-- dd:dd-q24 -->

### Problem 24 · independent

Write a function solve(z) that takes a 1-D PyTorch tensor of floats and returns a new tensor in which the largest value has been replaced by 0.0, with everything else unchanged. If the maximum occurs more than once, replace only its first occurrence. Do not modify the input tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2., 0., 4.])
```


In [ ]:
import torch as t

def solve(z):
    """Return z with its largest value replaced by 0.0."""
    return None


# Example run — the grader calls solve() with several vectors.
example = t.tensor([2.0, 9.0, 4.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(24)


In [ ]:
#@title 💡 Solution — Problem 24
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    out = z.clone()
    out[out.argmax()] = 0.0
    return out


example = t.tensor([2.0, 9.0, 4.0])
print(solve(example))


<!-- dd:dd-q61 -->

### Problem 61 · independent

Write a function solve(z, v) that takes a 1-D PyTorch numeric tensor z and a scalar v, and returns the single element of z that is closest to v (minimizing the absolute difference). If two elements are equally close, return the one at the lower index. Return a scalar value, not an tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(10)
```


In [ ]:
import torch as t

def solve(z, v):
    """Return the element of z nearest to the scalar v."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.tensor([0, 10, 20, 30])
print(solve(example, 12.4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(61)


In [ ]:
#@title 💡 Solution — Problem 61
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, v):
    return z[t.abs(z - v).argmin()]


example = t.tensor([0, 10, 20, 30])
print(solve(example, 12.4))


<!-- dd:dd-q168 -->

### Problem 168 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns, for each row, the column index of the row's largest value — but breaking ties by the LAST occurrence instead of argmax's usual first-occurrence rule.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2, 2])
```


In [ ]:
import torch as t

def solve(z):
    """Return each row's argmax with ties broken by the LAST occurrence."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 5, 5], [7, 2, 7]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(168)


In [ ]:
#@title 💡 Solution — Problem 168
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    rev = z.flip(1)
    return z.shape[1] - 1 - rev.argmax(dim=1)


print(solve(t.tensor([[1, 5, 5], [7, 2, 7]])))


#### Common mistakes

- **"argmax returns the maximum."** — It returns the *index* of the maximum.
  `v[v.argmax()]` is the value; needing both is common and costs one extra
  read.
- **"Ties are an error / unspecified."** — Ties resolve to the first
  occurrence, deterministically. Tasks that say "first occurrence" are
  describing argmin/argmax's default, not asking for extra work.
- **"Closest-to-target needs sorting."** — Sorting is O(n log n) and loses
  positions; `argmin(|v - t|)` is one pass and keeps them. Save sorting for
  when you need full order, not one winner.
